<a href="https://colab.research.google.com/github/CeZa18/ITAI-1371-ML-Labs/blob/main/L13_CesarZaldivar_ITAI1371.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 13 Lab - Building ML Pipelines
**Objective:** To understand the importance of `scikit-learn` **Pipelines** for creating robust, reproducible, and professional machine learning workflows.

**In this lab, you will refactor code from a previous lab into a clean, professional `Pipeline` object.**

## Part 1: Why Use Pipelines?
**Concept:** As you've seen, a typical ML workflow involves multiple steps: loading data, cleaning it, splitting it, preprocessing features (scaling, encoding), and finally, training a model. Managing all these steps separately can be messy and error-prone.

**Data Leakage:** A major risk of manual preprocessing is **data leakage**. This happens when information from the test set accidentally "leaks" into the training process. For example, if you calculate the mean for scaling using the *entire* dataset before splitting, the model has already "seen" the test data, leading to overly optimistic performance estimates.

**A `scikit-learn` Pipeline solves these problems by:**
1.  **Encapsulating** all workflow steps into a single object.
2.  **Preventing Data Leakage:** It ensures that preprocessing steps are fitted *only* on the training data during cross-validation or when calling `.fit()`.
3.  **Improving Reproducibility:** The entire workflow is saved as one object, making it easy to reuse and deploy.

## Part 2: The "Manual" Way (What We Did Before)
Let's revisit the Titanic dataset and the steps we took to prepare the data and train a model. This code should look familiar. Notice how many separate objects and steps there are.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load data
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Basic feature engineering and cleaning
'''
df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)
'''

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df.drop('Cabin', axis=1, inplace=True)
X = df.drop(['Survived', 'Name', 'Ticket', 'PassengerId'], axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Identify feature types
numeric_features = ['Age', 'Fare', 'SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex', 'Embarked']

# Manual Preprocessing
scaler = StandardScaler()
X_train_scaled_num = scaler.fit_transform(X_train[numeric_features])
X_test_scaled_num = scaler.transform(X_test[numeric_features])

# Note: using .transform() here!
encoder = OneHotEncoder(handle_unknown='ignore')
X_train_encoded_cat = encoder.fit_transform(X_train[categorical_features])
X_test_encoded_cat = encoder.transform(X_test[categorical_features])

# Combine preprocessed features
X_train_processed = np.hstack((X_train_scaled_num, X_train_encoded_cat.toarray()))
X_test_processed = np.hstack((X_test_scaled_num, X_test_encoded_cat.toarray()))

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_processed, y_train)
y_pred = model.predict(X_test_processed)
print(f"Accuracy (Manual Method): {accuracy_score(y_test, y_pred):.2%}")

Accuracy (Manual Method): 82.68%


## Part 3: The "Pipeline" Way
Now, let's do the exact same thing but encapsulate all the preprocessing steps into a single `Pipeline`.

**Your Task:** Use `make_pipeline` and `make_column_transformer` to build a complete workflow. This is the modern, professional way to build models in `scikit-learn`.

In [2]:
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

# Reload the data to start fresh
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.drop(['Cabin', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
X = df.drop('Survived', axis=1)
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- ENTER YOUR CODE HERE ---
# 1. Create a pipeline for numeric features
#    This pipeline will first impute missing 'Age' values with the median, then scale the features.
numeric_transformer = make_pipeline(
         SimpleImputer(strategy='median'),
         StandardScaler()
          )

# 2. Create a pipeline for categorical features
#    This pipeline will first impute missing 'Embarked' values with the most frequent value, then one-hot encode.
categorical_transformer = make_pipeline(
         SimpleImputer(strategy='most_frequent'),
         OneHotEncoder(handle_unknown='ignore')
          )

# 3. Use ColumnTransformer to apply different transformers to different columns
preprocessor = make_column_transformer(
         (numeric_transformer, ['Age', 'Fare', 'SibSp', 'Parch']),
         (categorical_transformer, ['Pclass', 'Sex', 'Embarked'])
          )

# 4. Create the final, full pipeline
#    This chains the preprocessor and the final model together.
final_pipeline = make_pipeline(
         preprocessor,
         RandomForestClassifier(random_state=42)
          )

# 5. Fit and evaluate the entire pipeline in one step!
final_pipeline.fit(X_train, y_train)
y_pred_pipeline = final_pipeline.predict(X_test)
print(f"Accuracy (Pipeline Method): {accuracy_score(y_test, y_pred_pipeline):.2%}")

Accuracy (Pipeline Method): 82.68%


## 📝 Reflective Knowledge Check
**Instructions:** Answer the following questions in this markdown cell.
1.  **Code Comparison:** Look at the "Manual Way" versus the "Pipeline Way". What are the three biggest advantages you see in using the Pipeline approach?
2.  **Data Leakage Explained:** In the manual code, we used `scaler.fit_transform()` on the training data but only `scaler.transform()` on the test data. Why was this distinction crucial? How does the Pipeline automatically handle this for you?
3.  **Extending the Pipeline:** Imagine you wanted to add a PCA step to reduce dimensionality *after* scaling and encoding but *before* the RandomForestClassifier. How would you modify your `final_pipeline` object to include this step? (You don't need to write the full code, just describe where you would add `PCA()`.)
4.  **Real-World Value:** You are handing your model over to another team to deploy into a web application. Why is giving them the single `final_pipeline` object much safer and more reliable than giving them the 5 separate objects (`scaler`, `encoder`, `model`, etc.) from the manual approach?

**[ENTER YOUR ANSWERS HERE]**

1.	(1) The pipeline removes the uncertainty of missed steps by always keeping all steps in the right sequence. (2) The code is cleaner and easier to modify by introducing modules for better reusability. (3) The Pipeline prevents data leakage by ensuring preprocessing steps are fitted only on training data, never on the test set.
2.	Fit is run only in the training step because its main function is to preprocess the data before the final process which runs transform on test, so that it prevents leakage of the test data during the training process. Pipelines encapsulate logic to enforce the golden rule: Fit on Train, Transform on Test. Because if we called `fit_transform()`on the test set, the scaler would learn the test set’s distribution (mean, std), meaning the model would have seen the test data – making accuracy falsely optimistic. And when you call `final_pipeline.fit(X_train)`, it fits all transformers on training data only. When we call `.predict(X_test)`, it automatically applies only `.transform()`, and we never have to think about it.
3.	I would add `PCA()` as a new step inside `final_pipeline`, between the `preprocessor` and the `RandomForestClassifier`, making it `make_pipeline(preprocessor, PCA(), RandomForestClassifier())`.
4.	With one pipeline file, there is only one source of truth a single serializable object. This gives us three main reasons: 1) No missing steps or wrong order: with 5 separate objects, the deployment team could apply them in the wrong order or skip one entirely (e.g., forget to scale before the model). 2) With 5 objects, the implementation team would have to manually run: `scale → encode → combine → predict`. With the pipeline: `final_pipeline.predict(raw_data)` — done. Fewer lines = fewer human errors. 3) No version mismatch. With separate objects, there’s a risk of pairing the wrong version of the scaler with the model (e.g., a scaler fitted on different data). The pipeline bundles everything that was trained together into one file — the scaler and model are always guaranteed to work.
